# 11 — Video Assembly

**Marker:** `NOTEBOOK_11_VIDEO_ASSEMBLY_FRESH_V1`

This notebook creates the finished vertical short.

It combines:

- one Subway Surfers gameplay recording
- `narration.wav` from Notebook 09
- `captions.ass` from Notebook 10
- metadata and timing information from the previous notebooks

Default behavior for iPhone portrait screen recordings:

- scale to fill 1080×1920
- keep the top of the recording
- crop extra height from the bottom
- burn in captions
- use narration as the final audio


## Load the project

In [ ]:
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "educational_shorts").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not find the project root containing educational_shorts/."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from educational_shorts.video import (
    find_background_video,
    find_ffmpeg_executable,
    find_ffprobe_executable,
    find_metadata_file,
    probe_video,
    render_video,
    summarize_render_manifest,
)

print("NOTEBOOK_11_VIDEO_ASSEMBLY_FRESH_V1")
print(f"Project root: {PROJECT_ROOT}")

## FFmpeg check

In [ ]:
ffmpeg = find_ffmpeg_executable()
ffprobe = find_ffprobe_executable()

print(f"ffmpeg: {ffmpeg}")
print(f"ffprobe: {ffprobe}")

## Configuration

In [ ]:
METADATA_DIRECTORY = PROJECT_ROOT / "data" / "metadata"
GAMEPLAY_DIRECTORY = PROJECT_ROOT / "data" / "gameplay" / "subway_surfers"
VIDEO_OUTPUT_DIRECTORY = PROJECT_ROOT / "data" / "videos"

# Leave as None to use the newest metadata file.
METADATA_FILENAME = None

# Leave as None to use the newest video found in GAMEPLAY_DIRECTORY.
BACKGROUND_VIDEO_FILENAME = None

OUTPUT_WIDTH = 1080
OUTPUT_HEIGHT = 1920

# "top" keeps the top of tall iPhone screen recordings and crops the bottom.
# Use "center" if the crop feels too high.
CROP_ANCHOR_Y = "top"

RANDOM_SEED = 42
DURATION_EXTRA_SECONDS = 0.25
START_PADDING_SECONDS = 1.0
END_PADDING_SECONDS = 1.0

# Keep this True while you only have one short gameplay clip.
LOOP_BACKGROUND = True

CRF = 20
PRESET = "medium"
AUDIO_BITRATE = "192k"

print(f"Metadata directory: {METADATA_DIRECTORY}")
print(f"Gameplay directory: {GAMEPLAY_DIRECTORY}")
print(f"Output directory: {VIDEO_OUTPUT_DIRECTORY}")

## Find metadata and gameplay video

In [ ]:
metadata_path = find_metadata_file(
    metadata_directory=METADATA_DIRECTORY,
    filename=METADATA_FILENAME,
)

background_video_path = find_background_video(
    gameplay_directory=GAMEPLAY_DIRECTORY,
    filename=BACKGROUND_VIDEO_FILENAME,
)

background_probe = probe_video(background_video_path)

print(f"Metadata: {metadata_path}")
print(f"Gameplay video: {background_video_path}")
print(f"Background duration: {background_probe.duration_seconds}s")
print(f"Background size: {background_probe.width}x{background_probe.height}")
print(f"Frame rate: {background_probe.frame_rate}")

## Render the final video

In [ ]:
render_manifest = render_video(
    project_root=PROJECT_ROOT,
    metadata_path=metadata_path,
    background_video_path=background_video_path,
    output_root=VIDEO_OUTPUT_DIRECTORY,
    output_width=OUTPUT_WIDTH,
    output_height=OUTPUT_HEIGHT,
    crop_anchor_y=CROP_ANCHOR_Y,
    random_seed=RANDOM_SEED,
    duration_extra_seconds=DURATION_EXTRA_SECONDS,
    start_padding_seconds=START_PADDING_SECONDS,
    end_padding_seconds=END_PADDING_SECONDS,
    loop_background=LOOP_BACKGROUND,
    crf=CRF,
    preset=PRESET,
    audio_bitrate=AUDIO_BITRATE,
)

for name, value in summarize_render_manifest(
    render_manifest
).items():
    print(f"{name}: {value}")

## Preview the final video

In [ ]:
from IPython.display import Video, display

output_video_path = (
    Path(render_manifest.output_directory)
    / render_manifest.output_video_filename
)

display(
    Video(
        filename=str(output_video_path),
        embed=False,
        width=360,
    )
)

print(f"Final video: {output_video_path}")

## Inspect render manifest

In [ ]:
print(render_manifest.model_dump_json(indent=2))